# 01 - What is RAG?

RAG stands for Retrieval Augmented Generation. It is a technique that lets a
large language model (LLM) answer questions using documents it has never seen:
your company wiki, your notes, your product manuals.

**What you will learn**

- Why an LLM cannot answer questions about your private data
- How giving the model the right text at the right time fixes that
- The three steps of RAG: retrieve, augment, generate

**Before you run this notebook**

Complete [SETUP.md](../SETUP.md): install the packages and put your OpenAI
API key in the .env file at the project root.

## The setup cell

Every notebook in this course starts with a cell like this. It loads your API
key from the .env file and creates a client object used to talk to OpenAI.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI()  # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"

print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


## The problem: LLMs do not know your data

An LLM is trained on public text from the internet, frozen at some point in
time. It knows a lot about the world, but it knows nothing about:

- private data (your company documents, your emails)
- anything that happened after its training ended

This course ships with documents about Aurora Dynamics, a fictional robotics
company (see the [data](../data/) folder). Because the company is made up, we
can be certain the model was never trained on it. Let us ask about it.

In [2]:
question = "How many days of paid annual leave do Aurora Dynamics employees get?"

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
)
print(response.choices[0].message.content)

I'm sorry, but I don't have specific information about the policies of Aurora Dynamics or any other specific company. To find out the number of days of paid annual leave for employees at Aurora Dynamics, it's best to refer to the company's official employee handbook, website, or contact their HR department directly.


The model either admits it does not know, or worse, makes up a
confident-sounding answer. Made up answers are called hallucinations, and they
are the main reason you cannot just point an LLM at questions about your data.

## The fix: put the answer in the prompt

LLMs are very good at reading. If the relevant text is inside the prompt, the
model answers correctly. Watch what happens when we paste the leave policy
document into the prompt ourselves.

In [3]:
with open("../data/03-leave-policy.txt") as f:
    leave_policy = f.read()

prompt = f"""Answer the question using only the context below.

Context:
{leave_policy}

Question: {question}"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
)
print(response.choices[0].message.content)

Aurora Dynamics employees receive 24 days of paid annual leave per calendar year.


A correct answer: 24 days. Nothing about the model changed. The only
difference is that the prompt now contains the facts needed to answer.

## The catch, and what RAG really is

We cheated: we knew which document held the answer, and our document was small
enough to paste whole. In real life you might have thousands of documents and
millions of words. You cannot paste everything into every prompt:

- prompts have a size limit (the context window)
- you pay per token you send
- burying the model in irrelevant text makes answers worse

So the real problem is: **for each question, automatically find the few
paragraphs that matter, and paste only those.** That is RAG:

```
 question
    |
    v
 1. RETRIEVE   search your documents for the most relevant pieces
    |
    v
 2. AUGMENT    build a prompt that contains the question + those pieces
    |
    v
 3. GENERATE   the LLM writes an answer grounded in the pieces
    |
    v
  answer (with sources)
```

The rest of this course builds each step by hand:

| Notebook | Step you build |
|----------|----------------|
| 02 | Talking to the LLM properly (generate) |
| 03 | Embeddings: how a computer measures "relevant" |
| 04 | Chunking: cutting documents into searchable pieces |
| 05 | Vector search: storing and finding pieces (retrieve) |
| 06 | The full pipeline (retrieve + augment + generate) |
| 07 | The same thing with the LangChain framework |

## Exercise

1. Ask the model (without context) about something else fictional from the
   data folder, for example: "What is the payload capacity of the Carrier X2
   robot?" Observe what it says.
2. Open [data/02-product-guide.txt](../data/02-product-guide.txt), paste its
   content into the prompt like we did above, and ask again.

Copy the two code cells above and modify them.

In [4]:
# Try the exercise here
